# 🚀 Vertex AI Prompt Optimizer (VAPO) for Radio Transcription

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/optimize_transcription_prompt.ipynb)


This notebook orchestrates the end-to-end server-side Prompt Tuning and Optimization (VAPO) pipeline for your radio transcription models. It compiles your training splits, aligns the dataset schemas, and launches tuning runs on Google's TPU servers.

### 🏁 Pipeline Phases:
1. **Setup & Authentication:** Configure your GCP project variables and authenticate your browser session.
2. **Dataset Compilation:** Reformat raw audio manifests into the `{input_text, target}` schema required by the optimizer.
3. **Optimization Launch:** Submit the server-side tuning job pre-configured with the correct `global` query routing.
4. **Monitoring & Administration:** Check active job states, view logs, and cancel jobs directly from this notebook.

In [ ]:
# @title 1. Install Dependencies & Bootstrap Environment
import sys
import os

# Detect if running in Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running inside Google Colab. Bootstrapping repository...")
    # Clone repository if not already present
    if not os.path.exists("radio-transcription"):
        !git clone -q https://github.com/watch-duty/radio-transcription.git

    # Install the model library in editable mode
    try:
        import common

        print("✅ Library 'common' already installed.")
    except ImportError:
        print("Installing library and dependencies...")
        # Install in editable mode
        %pip install -q -e radio-transcription/model
        import site
        import importlib

        importlib.reload(site)
        print("\n✅ Library 'common' installed successfully.")

# Install/upgrade the required SDKs and GCS library
%pip install -q --upgrade \
    "google-genai>=2.3,<3" \
    "google-cloud-aiplatform>=1.158.0,<2" \
    "google-cloud-storage" \
    "pydantic<=2.12.3" \
    etils \
    tqdm

In [ ]:
# @title 2. Imports
import json
import os
import subprocess
import sys
from google.cloud import storage
from google import genai
from google.genai import types
from google.colab import auth, userdata

# Import repository utilities
from common.gcs_utils import download_jsonl_manifest, upload_text
from common.manifest import is_scoreable_manifest_entry
from common.gemini.prompts import GEMINI_TRANSCRIBE_SYSTEM_PROMPT

In [ ]:
# @title 3. Configure Project Variables

# Retrieve credentials and buckets securely from Colab Secrets (userdata)
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET_NAME = userdata.get("GCS_BUCKET")
GCP_PROJECT_NUMBER = userdata.get("GCP_PROJECT_NUMBER")

assert GCP_PROJECT_ID, (
    "GCP_PROJECT_ID secret must be set in the Colab Secrets (userdata) panel."
)
assert GCS_BUCKET_NAME, (
    "GCS_BUCKET secret must be set in the Colab Secrets (userdata) panel."
)

GCP_LOCATION = "us-central1"

# @markdown ### 1. Training Manifest Source (for VAPO Tuning)
# @markdown Enter the GCS URI or bucket-relative path to your **training dataset** (e.g., `train.jsonl`).
# @markdown *⚠️ To prevent data leakage and evaluation contamination, point VAPO to your training split (`train.jsonl`), NOT your out-of-sample evaluation split (`eval.jsonl`).*
# fmt: off
INPUT_MANIFEST_URI = ""  # @param {type:"string"}
# fmt: on

assert INPUT_MANIFEST_URI, (
    "INPUT_MANIFEST_URI must be provided in the form above."
)


# Universal Path Normalization
def _normalize_gcs_uri(path_or_uri: str, bucket_name: str) -> str:
    path_or_uri = path_or_uri.strip()
    if path_or_uri.startswith("gs://"):
        return path_or_uri
    clean_path = path_or_uri.lstrip("/")
    return f"gs://{bucket_name}/{clean_path}"


gcs_input_manifest = _normalize_gcs_uri(INPUT_MANIFEST_URI, GCS_BUCKET_NAME)
gcs_manifest_dir = os.path.dirname(gcs_input_manifest)
print(f"✅ Resolved Manifest Path: {gcs_input_manifest}")
print(f"✅ Resolved Base Output Directory: {gcs_manifest_dir}")

# Auto-resolve Project Number via gcloud if not provided in secrets
if not GCP_PROJECT_NUMBER:
    try:
        project_number_raw = subprocess.check_output(
            [
                "gcloud",
                "projects",
                "describe",
                GCP_PROJECT_ID,
                "--format=value(projectNumber)",
            ]
        )
        GCP_PROJECT_NUMBER = project_number_raw.decode("utf-8").strip()
        print(f"✅ Resolved Project Number from GCP: {GCP_PROJECT_NUMBER}")
    except Exception as e:
        raise RuntimeError(
            f"Failed to resolve project number automatically for project '{GCP_PROJECT_ID}': {e}"
        ) from e
else:
    print(f"✅ Loaded Project Number from Secrets: {GCP_PROJECT_NUMBER}")

In [ ]:
# @title 4. Authenticate with GCP
# Run this cell to authenticate your browser session with Google Cloud
auth.authenticate_user()
print("✅ Browser session authenticated successfully!")

## 🔄 Phase 2: Dataset Reformatting & GCS Upload

The Vertex AI Prompt Optimizer requires the dataset to be in JSONL format, with the input mapped to `input_text` and the human ground-truth mapped **specifically to `"target"`**. This cell reformats your raw manifests and uploads them to GCS.

In [ ]:
# @title Compile and Upload Dataset

# Dynamically construct GCS output dataset path under dedicated vapo/ subdirectory
gcs_output_dataset = f"{gcs_manifest_dir}/vapo/apo_dataset.jsonl"

print("=== REFORMATTING DATASET FOR VERTEX AI PROMPT OPTIMIZER ===")
print(f"Source manifest GCS path: {gcs_input_manifest}")
print(f"Target dataset GCS path: {gcs_output_dataset}\n")

# 1. Download manifest in-memory
print("1. Downloading manifest from GCS...")
try:
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    manifest_entries = download_jsonl_manifest(
        storage_client, gcs_input_manifest
    )
    print("   ✅ Download successful!")
except Exception as e:
    raise RuntimeError(
        f"Failed to download manifest from {gcs_input_manifest}: {e}"
    ) from e

# 2. Reformat to schema
print("2. Reformatting entries into schema...")
compiled_count = 0
apo_rows = []
try:
    for entry in manifest_entries:
        if not is_scoreable_manifest_entry(entry):
            continue

        audio_uri = entry.get("audio_filepath")
        ground_truth_text = entry.get("text")

        # The VAPO schema maps input to 'input_text' and ground-truth to 'target'!
        apo_entry = {"input_text": audio_uri, "target": ground_truth_text}
        apo_rows.append(json.dumps(apo_entry))
        compiled_count += 1
    print(f"   ✅ Compiled {compiled_count} segments into schema.")
except Exception as e:
    raise RuntimeError(f"Failed to reformat manifest entries: {e}") from e

# 3. Upload back to GCS directly from memory
print("3. Uploading compiled dataset back to GCS...")
try:
    jsonl_content = "\n".join(apo_rows) + "\n"
    upload_text(
        storage_client,
        jsonl_content,
        gcs_output_dataset,
        content_type="application/jsonl",
    )
    print("   ✅ Upload successful! Dataset is 100% ready for tuning!")
except Exception as e:
    raise RuntimeError(
        f"Failed to upload compiled dataset to {gcs_output_dataset}: {e}"
    ) from e

## 🚀 Phase 3: Launch Prompt Optimization Job

Define your baseline jargon-heavy system prompt, construct the schema-compliant VAPO config, and submit the server-side job. 

> ⚠️ **CRITICAL ROUTING SOLUTION:** The target model location is set to **`"global"`** because Google has restricted early release models like `gemini-3.1-flash-lite` to the global endpoints. Setting it to `"global"` satisfies the container's region validation while successfully routing the queries!

In [ ]:
# @title Configure Baseline System Prompt for Optimization

# @markdown ### Select Your Seed Prompt Strategy:
# @markdown * **Custom / Experimental:** Default for research and testing new hypotheses (edit text block below).
# @markdown * **Production Baseline:** For continuous improvement against your deployed codebase (`prompts.py`).
PROMPT_SOURCE = "Custom / Experimental Prompt (edit below)"  # @param ["Custom / Experimental Prompt (edit below)", "Production Baseline (from prompts.py)"] {type:"string"}

# Edit your custom seed prompt directly between the triple quotes below:
# (Pre-populated with your champion baseline prompt with terminology for easy editing)
experimental_system_prompt = """\
Your primary task is to produce a strict, verbatim transcription of the spoken audio from fire, police, and EMS radio traffic.

CRITICAL RULES:
1. Output the transcript strictly and precisely as spoken in the audio, with no newlines. Do not add, invent, or infer any speech that is not clearly audible.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. If the audio contains a unit identifier, format it as the unit type followed by digits (e.g., Engine 41, Battalion 2). Apply this rule strictly only if the unit identifier is clearly spoken AND the context is unequivocally fire-related dispatch.

EXPECTED TERMINOLOGY (Domain Glossary):
Watch Duty, Cal Fire, USFS, Air Attack, Helitack, VLAT, SEAT, Air Tanker, Lead Plane, Strike Team, Task Force, Battalion, Division, Branch, Group, Staging, Incident Commander, IC, IC Overhead, Dispatch, Comm Center, Repeater, Tone Out, ROS (Rate of Spread), Spotting, Slop Over, Flank, Heel, Head, Containment, Control Line, Dozer Line, Hand Line, Hose Lay, Wetting, Retardant, Phos-Chek, Drop, Tag, Orbit, Reload, Return, En Route, On Scene, Available, In Quarters, Out of Service, Medevac, LZ (Landing Zone), ETA, PAR (Personnel Accountability Report), LCES, Red Flag, Weather, Wind Shift, Humidity, Temp, Fuel Moisture, Snag, Spot Fire, Structure Protection, Evacuation, Order, Warning, Road Closure, Traffic Control, staging area, command post, base camp, helibase.

QUALITY GATE:
If the audio is completely unintelligible, consists purely of RF noise/static, or lacks any human speech, you MUST output ONLY the exact token:
[UNINTELLIGIBLE]
"""

if PROMPT_SOURCE.startswith("Custom"):
    baseline_prompt = experimental_system_prompt.strip()
    print("🌟 MODE: TUNING CUSTOM / EXPERIMENTAL PROMPT")
else:
    baseline_prompt = GEMINI_TRANSCRIBE_SYSTEM_PROMPT
    print("🌟 MODE: TUNING PRODUCTION BASELINE PROMPT (`prompts.py`)")

print("=" * 60)
print(baseline_prompt)
print("=" * 60)

In [ ]:
# @title Submit Server-Side Prompt Optimization Job

assert "baseline_prompt" in globals(), (
    "Please run the 'Configure Baseline System Prompt' cell above before submitting!"
)

# @markdown ### Optimization Dataset Limit
# @markdown Max training rows to process during optimization (default 50 for fast tuning sweeps, set to 0 to process full dataset):
DATA_LIMIT = 50  # @param {type:"integer"}

# Dynamically link training dataset and output directory based on your resolved manifest directory!
train_dataset_uri = f"{gcs_manifest_dir}/vapo/apo_dataset.jsonl"
gcs_output_prefix = f"{gcs_manifest_dir}/vapo_outputs"

# 1. Construct VAPO configuration
vapo_data_settings = {
    "system_instruction": baseline_prompt,
    # Mark {input_text} as multimodal GCS audio URI
    "prompt_template": "{input_text} @@@audio/flac\n{target}",
    "target_model": "gemini-3.1-flash-lite",  # Target production model
    "thinking_budget": 0,
    "optimization_mode": "instruction",  # Optimize instruction text strictly (no static demo selection)
    "eval_metrics_types": ["bleu", "rouge_l"],
    "eval_metrics_weights": [0.5, 0.5],
    "input_data_path": train_dataset_uri,
    "output_path": f"{gcs_output_prefix}/results/",
    "project": GCP_PROJECT_ID,
    "num_steps": 10,
    # Global Routing Solution
    "target_model_location": "global",
    "optimizer_model_location": "global",
    "has_multimodal_inputs": True,
    "data_limit": DATA_LIMIT,
}

# 2. Upload config.json to GCS
config_gcs_path = f"{gcs_output_prefix}/config.json"
print(f"1. Uploading JSON configuration to GCS: {config_gcs_path}...")
storage_client = storage.Client(project=GCP_PROJECT_ID)
upload_text(
    storage_client,
    json.dumps(vapo_data_settings, indent=2),
    config_gcs_path,
    content_type="application/json",
)
print("   ✅ Configuration uploaded successfully!")

# 3. Initialize Vertex AI client using the google-genai SDK (enterprise Vertex AI mode)
print(f"\n2. Initializing Vertex AI Client in {GCP_LOCATION}...")
client = genai.Client(
    enterprise=True, project=GCP_PROJECT_ID, location=GCP_LOCATION
)

# 4. Launch the optimization job
print("\n3. Submitting Server-Side Prompt Optimization Job...")
service_account = f"{GCP_PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
vapo_run_config = {
    "config_path": config_gcs_path,
    "wait_for_completion": False,  # Submit and return immediately
    "service_account": service_account,
}

try:
    import warnings

    # Note: Using Vertex AI VAPO API client extensions (internal Prompts/PromptOptimizerMethod)
    from vertexai._genai.prompts import Prompts
    from vertexai._genai.types import PromptOptimizerMethod

    warnings.filterwarnings("ignore", category=UserWarning)

    prompts_client = Prompts(client._api_client)
    result = prompts_client.launch_optimization_job(
        method=PromptOptimizerMethod.VAPO, config=vapo_run_config
    )
    print("\n==================================================")
    print("🎉 SUCCESS! PROMPT OPTIMIZER JOB SUBMITTED!")
    print("==================================================")
    print(f"Manifest Directory: {gcs_manifest_dir}")
    print(f"Service Account:    {service_account}")
    console_url = f"https://console.cloud.google.com/vertex-ai/training/custom-jobs?project={GCP_PROJECT_ID}"
    print("\n🔗 Monitor Running Job in Cloud Console:")
    print(f"   👉 {console_url}")
    print("==================================================\n")
except Exception as e:
    print(f"❌ Failed to submit job: {e}")
    raise RuntimeError(f"VAPO job submission failed: {e}") from e

## 🕵️‍♂️ Phase 4: Monitoring and Job Administration

You can monitor your running Custom Jobs directly in the Google Cloud Console web UI:
👉 **[Open Vertex AI Custom Jobs Console](https://console.cloud.google.com/vertex-ai/training/custom-jobs)** *(Ensure your project selector at the top is accurate and region filter is set to `us-central1`)*

Alternatively, use the cells below to query the queue, check specific job states, or cancel unwanted runs directly from the notebook.

In [ ]:
# @title 1. List Recent Custom Jobs
# Query the 4 most recent Custom Jobs in us-central1
!gcloud ai custom-jobs list \
    --project={GCP_PROJECT_ID} \
    --region={GCP_LOCATION} \
    --limit=4 \
    --format="table(name.basename():label=JOB_ID, state:label=STATUS, createTime:label=CREATED_TIME)"

In [ ]:
# @title 2. Describe Specific Job State
TARGET_JOB_ID = ""  # @param {type:"string"}

if TARGET_JOB_ID:
    !gcloud ai custom-jobs describe {TARGET_JOB_ID} \
        --project={GCP_PROJECT_ID} \
        --region={GCP_LOCATION} \
        --format="value(state)"
else:
    print("⚠️ Please enter a TARGET_JOB_ID in the form parameter above!")

In [ ]:
# @title 3. Cancel Running Custom Job
CANCEL_JOB_ID = ""  # @param {type:"string"}

if CANCEL_JOB_ID:
    confirm = input(
        f"Are you absolutely sure you want to CANCEL job {CANCEL_JOB_ID}? (y/n): "
    )
    if confirm.lower() == "y":
        !gcloud ai custom-jobs cancel {CANCEL_JOB_ID} \
            --project={GCP_PROJECT_ID} \
            --region={GCP_LOCATION} \
            --quiet
        print(f"✅ Cancel request sent for job {CANCEL_JOB_ID}!")
    else:
        print("Cancellation aborted.")
else:
    print("⚠️ Please enter a CANCEL_JOB_ID in the form parameter above!")

In [ ]:
# @title 4. Print the Final Optimized System Instruction
# Run this cell to instantly read GCS and print the final winning prompt!
import json
from etils import epath

# Construct the path to the final optimized results file based on your manifest directory
optimized_results_path = f"{gcs_manifest_dir}/vapo_outputs/results/instruction/demonstration/optimized_results.json"
print(f"Reading final optimized results from: {optimized_results_path}\n")

try:
    with epath.Path(optimized_results_path).open("r") as f:
        data = json.load(f)

    winning_prompt = data.get("prompt", "")
    print("==================================================")
    print("🏆 WINNING SYSTEM INSTRUCTION:")
    print("==================================================")
    print(winning_prompt)
    print("==================================================\n")
except Exception as e:
    print(f"❌ Failed to load optimized prompt: {e}")
    print("Ensure the job has completed and the path is correct.")

## 📊 Phase 5: Visualize Results with Native Colab Tables

This section uses Colab's native interactive data tables to inspect your mutated system instructions and segment-level BLEU/ROUGE score improvements without any external web servers, reverse tunnels, or WebSocket timeouts.

In [ ]:
# @title 1. View VAPO Results (Native Colab Interactive Tables)
import io
import json
import os
from google.cloud import storage
import pandas as pd

# Enable Colab's native interactive data tables (search, sort, pagination)
try:
    from google.colab import data_table

    data_table.enable_dataframe_formatter()
    print("✅ Enabled native Colab interactive data tables.")
except ImportError:
    pass

# Auto-resolve the results directory to point directly to your completed run!
default_results_dir = (
    f"{gcs_manifest_dir}/vapo_outputs/results/instruction/demonstration"
)
RESULTS_GCS_DIR = ""  # @param {type:"string"}

if not RESULTS_GCS_DIR:
    RESULTS_GCS_DIR = default_results_dir

print(f"📡 Fetching VAPO results from:\n   👉 {RESULTS_GCS_DIR}\n")


def load_gcs_json(gcs_uri: str):
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket_name = gcs_uri.split("/")[2]
    blob_path = "/".join(gcs_uri.split("/")[3:])
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    return json.loads(blob.download_as_text())


try:
    templates_data = load_gcs_json(f"{RESULTS_GCS_DIR}/templates.json")
    eval_data = load_gcs_json(f"{RESULTS_GCS_DIR}/eval_results.json")

    # 1. Mutated System Instructions Table
    templates_df = pd.json_normalize(templates_data)
    print("==================================================")
    print(f"🧠 MUTATED SYSTEM INSTRUCTIONS ({len(templates_df)} Iterations):")
    print("==================================================")
    display(templates_df)

    # 2. Segment-Level Evaluation Metrics (Latest / Winning Iteration)
    latest_eval_table_str = eval_data[-1]["metrics_table"]
    eval_df = pd.read_json(io.StringIO(latest_eval_table_str))

    # Filter to most relevant columns for clean scannability
    cols_to_show = [
        col
        for col in eval_df.columns
        if any(
            k in col.lower()
            for k in [
                "target",
                "response",
                "score",
                "bleu",
                "rouge",
                "explanation",
            ]
        )
    ]
    if cols_to_show:
        eval_df = eval_df[cols_to_show]

    print("\n==================================================")
    print(
        f"📊 SEGMENT-LEVEL EVALUATION METRICS (Winning Iteration, {len(eval_df)} segments):"
    )
    print("==================================================")
    display(eval_df)

except Exception as e:
    print(f"❌ Failed to load results from GCS: {e}")
    print(
        "💡 Tip: Verify that RESULTS_GCS_DIR points to a completed VAPO run folder (e.g., .../results/instruction/demonstration)."
    )